<a href="https://colab.research.google.com/github/Shakada26/carisurg-portfolio/blob/main/Week-7/Week7_complex_model_and_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7: Complex Model and Six-Axis Benchmark (Interim)

**CariSurg MedTech Pathways · Mercer General Hospital · ED triage (ESI 1–5)**

This notebook continues directly from the Week 5–6 notebook: same target, same feature set,
same preprocessing, same 80/20 split (seed 42). The only thing that changes is the model.
It produces (1) a trained complex model, (2) initial per-class benchmark metrics, and
(3) a draft comparison table written to ../docs/week7_benchmark_table.md.

### Reproducibility (identical to Week 6)
- **Seed:** 42, used for the split and every model.
- **Target:** Triage_Level (ESI 1–5, derived from esi).
- **Leakage removed:** esi (raw target), disposition, previousdispo (post-triage), Unnamed: 0 (index).
- **Features:** every remaining column - vitals, chief-complaint flags, and demographic/admin fields.
- **Preprocessing:** ColumnTransformer - one-hot encode text columns, standardise numeric columns.
- **Split:** train_test_split(test_size=0.2, stratify=y, random_state=42).

In [2]:
# ---- libraries ----
import time, os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report)
import joblib

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 140)
print("Libraries loaded. Seed =", SEED)

Libraries loaded. Seed = 42


## 1 · Load the data and rebuild the target


In [4]:
DATA_PATH = "/content/drive/MyDrive/yaleemmlc_admissionprediction_triage.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "yaleemmlc_admissionprediction_triage.csv"   # local fallback

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns from: {DATA_PATH}")

# Build the target exactly as in Week 6
df["esi"] = pd.Categorical(df["esi"].astype(int), categories=[1, 2, 3, 4, 5], ordered=True)
df["Triage_Level"] = df["esi"].astype(int)

# Anything that leaks the answer or is a bare index
leakage_cols = ["esi", "disposition", "previousdispo", "Unnamed: 0"]

df["Triage_Level"].value_counts().sort_index()

Loaded 55,121 rows x 226 columns from: /content/drive/MyDrive/yaleemmlc_admissionprediction_triage.csv


,count
Triage_Level,
1,77
2,17924
3,27010
4,8896
5,1214


## 2 · Features (X), target (y), and the Week 6 preprocessor
X is every column except the target and leakage columns. The ColumnTransformer one-hot encodes
text columns and standardises numeric ones. sparse_output=False keeps the matrix dense so the
gradient-boosting model can consume it.

In [5]:
X = df.drop(columns=["Triage_Level"] + [c for c in leakage_cols if c in df.columns])
y = df["Triage_Level"]

categorical_columns = X.select_dtypes(include="object").columns
numeric_columns = X.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
        ("numerical",   StandardScaler(), numeric_columns),
    ]
)
print("X columns:", X.shape[1], "| categorical:", len(categorical_columns), "| numeric:", len(numeric_columns))

X columns: 222 | categorical: 13 | numeric: 209


## 3 · Reuse the EXACT Week 6 split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print("train:", X_train.shape[0], "| test:", X_test.shape[0])
print("test-set Triage_Level mix:\n", y_test.value_counts(normalize=True).sort_index().round(3))

train: 44096 | test: 11025
test-set Triage_Level mix:
 Triage_Level
1    0.001
2    0.325
3    0.490
4    0.161
5    0.022
Name: proportion, dtype: float64


## 4 · The models
Two Week 6 baselines (rebuilt here) plus the complex models, all in the same preprocessing pipeline.
The Random Forest uses class_weight="balanced" to protect the rare ESI-1 class; the baselines are
left exactly as in Week 6 (no class weighting) to keep the comparison honest.

In [7]:
CLASSES = [1, 2, 3, 4, 5]

def make_pipe(model):
    return Pipeline(steps=[("preprocessing", preprocessor), ("model", model)])

def benchmark(name, pipe, fit_kwargs=None):
    fit_kwargs = fit_kwargs or {}
    t0 = time.perf_counter(); pipe.fit(X_train, y_train, **fit_kwargs); train_s = time.perf_counter() - t0
    reps = 3; t0 = time.perf_counter()
    for _ in range(reps): preds = pipe.predict(X_test)
    infer_ms = (time.perf_counter() - t0) / reps / len(X_test) * 1000.0
    rep = classification_report(y_test, preds, labels=CLASSES, output_dict=True, zero_division=0)
    row = {"model": name,
           "accuracy":        round(accuracy_score(y_test, preds), 4),
           "precision_macro": round(precision_score(y_test, preds, average="macro", zero_division=0), 4),
           "recall_macro":    round(recall_score(y_test, preds, average="macro", zero_division=0), 4),
           "f1_macro":        round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
           "recall_ESI1":     round(rep["1"]["recall"], 4),
           "train_time_s":    round(train_s, 4),
           "infer_ms_per_pred": round(infer_ms, 4)}
    return row, rep, pipe

results, perclass, fitted = [], {}, {}
def run(name, model, fit_kwargs=None):
    row, rep, pipe = benchmark(name, make_pipe(model), fit_kwargs)
    results.append(row); perclass[name] = rep; fitted[name] = pipe
    print(f"{name:26s} macroF1={row['f1_macro']:.4f}  recall(ESI1)={row['recall_ESI1']:.4f}  "
          f"train={row['train_time_s']:.3f}s  infer={row['infer_ms_per_pred']:.4f}ms")
    return pipe

# --- baselines (exactly as Week 6) ---
run("Dummy (stratified)",       DummyClassifier(strategy="stratified", random_state=SEED))
run("LogReg (baseline)",        LogisticRegression(max_iter=1000, random_state=SEED))
run("Decision Tree (baseline)", DecisionTreeClassifier(max_depth=5, random_state=SEED))

# --- complex: Random Forest (class-balanced) ---
run("Random Forest",
    RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1))

# --- complex: Gradient Boosting (class_weight if supported, else balanced sample_weight) ---
try:
    run("Gradient Boosting",
        HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1, max_iter=300,
                                       class_weight="balanced", random_state=SEED))
except TypeError:
    sw = compute_sample_weight("balanced", y_train)
    run("Gradient Boosting",
        HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1, max_iter=300, random_state=SEED),
        fit_kwargs={"model__sample_weight": sw})

Dummy (stratified)         macroF1=0.2039  recall(ESI1)=0.0000  train=0.790s  infer=0.0135ms
LogReg (baseline)          macroF1=0.5266  recall(ESI1)=0.2500  train=50.248s  infer=0.0150ms
Decision Tree (baseline)   macroF1=0.2856  recall(ESI1)=0.0000  train=2.756s  infer=0.0194ms
Random Forest              macroF1=0.3742  recall(ESI1)=0.0000  train=65.232s  infer=0.0969ms
Gradient Boosting          macroF1=0.4443  recall(ESI1)=0.4375  train=16.141s  infer=0.0784ms


## 5 · Six-axis benchmark table

In [8]:
bench = pd.DataFrame(results).set_index("model")
bench = bench[["accuracy","precision_macro","recall_macro","f1_macro","recall_ESI1","train_time_s","infer_ms_per_pred"]]
bench

,accuracy,precision_macro,recall_macro,f1_macro,recall_ESI1,train_time_s,infer_ms_per_pred
model,,,,,,,
Dummy (stratified),0.3754,0.2041,0.2037,0.2039,0.0000,0.7904,0.0135
LogReg (baseline),0.7063,0.5882,0.5011,0.5266,0.2500,50.2482,0.0150
Decision Tree (baseline),0.5573,0.3533,0.2890,0.2856,0.0000,2.7564,0.0194
Random Forest,0.6573,0.5129,0.3594,0.3742,0.0000,65.2321,0.0969
Gradient Boosting,0.5936,0.4279,0.5826,0.4443,0.4375,16.1410,0.0784


## 6 · Per-class metrics (not just aggregates)
Aggregate scores hide the rare classes. ESI 1 (sickest, rarest) is the row that governs the decision.

In [9]:
def per_class_frame(metric):
    data = {name: {f"ESI {k}": round(perclass[name][str(k)][metric], 4) for k in CLASSES} for name in perclass}
    return pd.DataFrame(data).T

print("=== RECALL per class ==="); display(per_class_frame("recall"))
print("=== PRECISION per class ==="); display(per_class_frame("precision"))
print("=== F1 per class ==="); display(per_class_frame("f1-score"))

=== RECALL per class ===


,ESI 1,ESI 2,ESI 3,ESI 4,ESI 5
Dummy (stratified),0.0000,0.3275,0.4981,0.1478,0.0453
LogReg (baseline),0.2500,0.6753,0.7686,0.6594,0.1523
Decision Tree (baseline),0.0000,0.2510,0.8612,0.3328,0.0000
Random Forest,0.0000,0.6109,0.8238,0.3378,0.0247
Gradient Boosting,0.4375,0.7498,0.4574,0.6965,0.5720


=== PRECISION per class ===


,ESI 1,ESI 2,ESI 3,ESI 4,ESI 5
Dummy (stratified),0.0000,0.3337,0.4908,0.1471,0.0487
LogReg (baseline),0.4000,0.7447,0.7071,0.6488,0.4405
Decision Tree (baseline),0.0000,0.7745,0.5448,0.4471,0.0000
Random Forest,0.0000,0.7242,0.6277,0.6670,0.5455
Gradient Boosting,0.0959,0.6391,0.8005,0.4444,0.1596


=== F1 per class ===


,ESI 1,ESI 2,ESI 3,ESI 4,ESI 5
Dummy (stratified),0.0000,0.3306,0.4944,0.1475,0.0469
LogReg (baseline),0.3077,0.7083,0.7366,0.6540,0.2263
Decision Tree (baseline),0.0000,0.3792,0.6674,0.3816,0.0000
Random Forest,0.0000,0.6627,0.7125,0.4485,0.0472
Gradient Boosting,0.1573,0.6900,0.5822,0.5426,0.2496


## 7 · Interpretability - explaining one prediction to Dr Reyes
Global feature importances (mapped back through the one-hot/scaler step) plus, for any single
patient, the model's class probabilities.

In [10]:
rf_pipe = fitted["Random Forest"]
feat_names = rf_pipe.named_steps["preprocessing"].get_feature_names_out()
importances = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=feat_names).sort_values(ascending=False)
print("Top 12 features (Random Forest):")
display(importances.head(12).round(4).to_frame("importance"))

i = 0
patient = X_test.iloc[[i]]
pred = rf_pipe.predict(patient)[0]
proba = rf_pipe.predict_proba(patient)[0]
print(f"\nPatient #{i}: predicted Triage_Level = {pred} (true = {y_test.iloc[i]})")
print("Class probabilities:", dict(zip(rf_pipe.named_steps['model'].classes_, proba.round(3))))

Top 12 features (Random Forest):


,importance
numerical__triage_vital_sbp,0.0537
numerical__age,0.0517
numerical__triage_vital_dbp,0.0509
numerical__triage_vital_hr,0.0476
numerical__triage_glucose,0.0459
numerical__triage_vital_temp,0.0432
numerical__triage_vital_rr,0.0309
numerical__triage_vital_o2,0.0307
categorical__arrivalmode_ambulance,0.0228
numerical__cc_strokealert,0.0224



Patient #0: predicted Triage_Level = 2 (true = 2)
Class probabilities: {np.int64(1): np.float64(0.0), np.int64(2): np.float64(0.553), np.int64(3): np.float64(0.403), np.int64(4): np.float64(0.04), np.int64(5): np.float64(0.003)}


In [11]:
os.makedirs("../docs", exist_ok=True)
os.makedirs("../models", exist_ok=True)

def df_to_md(frame):
    cols = ["model"] + list(frame.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"]*len(cols)) + "|"]
    for name, r in frame.iterrows():
        vals = [str(name)] + [f"{v:.4f}" if isinstance(v, float) else str(v) for v in r]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

md_table = df_to_md(bench)
header = ("# Week 7 — Draft Benchmark Table\n\n"
          f"Dataset: Mercer/Yale ED triage · target = Triage_Level (ESI 1-5) · seed = {SEED} · "
          "same 80/20 stratified split and preprocessing as Week 6.\n\n"
          "Metrics macro-averaged unless suffixed; recall_ESI1 is the primary clinical metric. "
          "Train time in seconds; inference time in ms per single prediction.\n\n")
with open("../docs/week7_benchmark_table.md", "w") as f:
    f.write(header + md_table + "\n")
per_class_frame("recall").to_csv("../docs/week7_perclass_recall.csv")
for key, fname in [("Random Forest","w7_random_forest"), ("Gradient Boosting","w7_gradient_boosting"),
                   ("LogReg (baseline)","w7_logreg"), ("Decision Tree (baseline)","w7_tree")]:
    joblib.dump(fitted[key], f"../models/{fname}.joblib")
print("Wrote ../docs/week7_benchmark_table.md and saved models to ../models/\n")
print(md_table)

Wrote ../docs/week7_benchmark_table.md and saved models to ../models/

| model | accuracy | precision_macro | recall_macro | f1_macro | recall_ESI1 | train_time_s | infer_ms_per_pred |
|---|---|---|---|---|---|---|---|
| Dummy (stratified) | 0.3754 | 0.2041 | 0.2037 | 0.2039 | 0.0000 | 0.7904 | 0.0135 |
| LogReg (baseline) | 0.7063 | 0.5882 | 0.5011 | 0.5266 | 0.2500 | 50.2482 | 0.0150 |
| Decision Tree (baseline) | 0.5573 | 0.3533 | 0.2890 | 0.2856 | 0.0000 | 2.7564 | 0.0194 |
| Random Forest | 0.6573 | 0.5129 | 0.3594 | 0.3742 | 0.0000 | 65.2321 | 0.0969 |
| Gradient Boosting | 0.5936 | 0.4279 | 0.5826 | 0.4443 | 0.4375 | 16.1410 | 0.0784 |
